# DataCo supply chain analysis
**ISIT312 Big Data Management · Member 3**

Which shipping modes have the highest late-order rates? Which categories earn the most profit on completed orders? DataCo Global provides the public records for this demonstration. Caspian Distribution FZE is the business case.

**Run the notebook.** Extract the project ZIP, open this notebook in Jupyter and choose Run All with Python 3 and pandas installed. Keep the supplied `DataCo_Original_Files.zip` beside the notebook. In Google Colab you can also upload just this notebook and choose Run all. If the CSVs and archive are absent, the first cell downloads the public archive. It checks all three files against Mendeley version 5. The dictionary is written to the current working folder.

Source: F. Constante, F. Silva and A. Pereira (2019), *DataCo SMART SUPPLY CHAIN FOR BIG DATA ANALYSIS*, V5. https://doi.org/10.17632/8gx2fvg2k6.5

In [1]:
from pathlib import Path
from zipfile import ZipFile
from urllib.request import Request, urlopen
from io import BytesIO
import hashlib
import shutil
import pandas as pd

files = {
    'DataCoSupplyChainDataset.csv': 'fa6d022ed437155e1a2f0378710602848703c8a7f203f7ff5d77805bf8480aa6',
    'DescriptionDataCoSupplyChain.csv': '9828e34669bd6d77e3b4463364cc44a5d52446b5e246fc258758cfe592566c4b',
    'tokenized_access_logs.csv': '7a4372df63a1e87f5add68bc036bb6064db09069998f25a679aec39f1a8d7765',
}
raw_dir = next((p for p in [Path('raw'), Path('.')] if all((p / name).is_file() for name in files)), None)
archive = Path('DataCo_Original_Files.zip')
if raw_dir is None and not archive.exists():
    url = 'https://www.kaggle.com/api/v1/datasets/download/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis'
    temp = archive.with_suffix('.part')
    with urlopen(Request(url, headers={'User-Agent': 'ISIT312 dataset analysis'}), timeout=120) as response, temp.open('wb') as dest:
        shutil.copyfileobj(response, dest)
    with ZipFile(temp) as z:
        assert set(files).issubset(z.namelist()) and z.testzip() is None
    temp.replace(archive)

def read_source(name, **kwargs):
    if raw_dir is None:
        with ZipFile(archive) as z:
            data = z.read(name)
    else:
        data = (raw_dir / name).read_bytes()
    assert hashlib.sha256(data).hexdigest() == files[name], f'File differs from Mendeley V5: {name}'
    return pd.read_csv(BytesIO(data), encoding='latin1', keep_default_na=False, na_values=[''], **kwargs)

lines = read_source('DataCoSupplyChainDataset.csv', dtype={'Customer Zipcode':'string', 'Order Zipcode':'string'})
guide = read_source('DescriptionDataCoSupplyChain.csv')
logs = read_source('tokenized_access_logs.csv')
print('Verified all three original CSV files against their published SHA-256 values.')

Verified all three original CSV files against their published SHA-256 values.


## 1. What a row represents
A customer can buy several items in one order. Compare the two IDs before counting orders.

In [2]:
items_per_order = lines.groupby('Order Id').size()
print(f'Order item rows: {len(lines):,} | Columns: {lines.shape[1]}')
print(f'Unique order item IDs: {lines["Order Item Id"].nunique():,}')
print(f'Unique order IDs: {len(items_per_order):,}')
print(f'Orders with multiple items: {(items_per_order > 1).sum():,}')
print(f'Items per order: {items_per_order.min()} to {items_per_order.max()}')
print(f'Products: {lines["Product Card Id"].nunique()} | Category IDs: {lines["Category Id"].nunique()} | Destination regions: {lines["Order Region"].nunique()}')

Order item rows: 180,519 | Columns: 53
Unique order item IDs: 180,519
Unique order IDs: 65,752
Orders with multiple items: 45,902
Items per order: 1 to 5
Products: 118 | Category IDs: 51 | Destination regions: 23


## 2. Missing values and duplicate records
Count empty and whitespace-only cells. An exact duplicate matches all 53 columns. Keep ZIP codes as text.

In [3]:
missing = lines.isna().sum()
for col in lines.select_dtypes(include=['object', 'string']).columns:
    missing[col] += lines[col].astype('string').str.strip().eq('').sum()
quality = pd.DataFrame({'missing': missing, 'percent': (100 * missing / len(lines)).round(4)})
print(quality.loc[missing > 0].sort_values('missing', ascending=False).to_string())
print(f'Exact duplicate rows: {lines.duplicated().sum():,}')
print(f'Repeated item IDs after the first occurrence: {lines["Order Item Id"].duplicated().sum():,}')
constant = lines.nunique(dropna=False)
print('Constant or wholly empty fields: ' + ', '.join(constant[constant == 1].index))

                     missing   percent
Product Description   180519  100.0000
Order Zipcode         155679   86.2397
Customer Lname             8    0.0044
Customer Zipcode           3    0.0017
Exact duplicate rows: 0
Repeated item IDs after the first occurrence: 0
Constant or wholly empty fields: Customer Email, Customer Password, Product Description, Product Status


## 3. Dates and field definitions
The raw date strings follow month/day/year and a 24-hour clock. Validate aliases before choosing one preferred field for an analysis.

In [4]:
fmt = '%m/%d/%Y %H:%M'
ordered = pd.to_datetime(lines['order date (DateOrders)'], format=fmt, errors='coerce')
shipped = pd.to_datetime(lines['shipping date (DateOrders)'], format=fmt, errors='coerce')
for title, series in [('Order date', ordered), ('Shipping date', shipped)]:
    print(f'{title}: {series.min()} through {series.max()} | unparsed: {series.isna().sum()}')
print(f'Shipping date precedes order date: {(shipped < ordered).sum()}')
pairs = [('Benefit per order', 'Order Profit Per Order'), ('Sales per customer', 'Order Item Total'),
         ('Order Item Cardprod Id', 'Product Card Id'), ('Category Id', 'Product Category Id'),
         ('Customer Id', 'Order Customer Id')]
for left, right in pairs:
    print(f'{left} vs {right}: {lines[left].ne(lines[right]).sum()} differences')
category_keys = lines[['Category Id', 'Category Name']].drop_duplicates()
print('Names used for multiple category IDs:')
print(category_keys[category_keys.duplicated('Category Name', keep=False)].sort_values('Category Id').to_string(index=False))
print(f'Orders where profit differs across items: {(lines.groupby("Order Id")["Order Profit Per Order"].nunique() > 1).sum():,}')
print(f'Items with negative profit: {lines["Order Profit Per Order"].lt(0).sum():,}')

Order date: 2015-01-01 00:00:00 through 2018-01-31 23:38:00 | unparsed: 0
Shipping date: 2015-01-03 00:00:00 through 2018-02-06 22:14:00 | unparsed: 0
Shipping date precedes order date: 0
Benefit per order vs Order Profit Per Order: 0 differences
Sales per customer vs Order Item Total: 0 differences
Order Item Cardprod Id vs Product Card Id: 0 differences
Category Id vs Product Category Id: 0 differences
Customer Id vs Order Customer Id: 0 differences
Names used for multiple category IDs:
 Category Id Category Name
          13   Electronics
          37   Electronics
Orders where profit differs across items: 45,902
Items with negative profit: 33,784


## 4. Late orders by shipping mode
Each order gets one vote. Check that the mode, status and flag agree across all its items before reducing to one row per order. Use orders with a recorded late, on-time or early outcome in the denominator. Exclude shipping cancellations.

In [5]:
delivery_fields = ['Shipping Mode', 'Delivery Status', 'Late_delivery_risk']
assert lines.groupby('Order Id')[delivery_fields].nunique(dropna=False).eq(1).all().all()
orders = lines.drop_duplicates('Order Id')
eligible = orders[orders['Delivery Status'].isin(['Advance shipping','Shipping on time','Late delivery'])].copy()
assert len(eligible) + orders['Delivery Status'].eq('Shipping canceled').sum() == len(orders)
assert eligible['Delivery Status'].eq('Late delivery').eq(eligible['Late_delivery_risk'].eq(1)).all()
assert eligible['Late_delivery_risk'].eq(eligible['Days for shipping (real)'].gt(eligible['Days for shipment (scheduled)']).astype(int)).all()
rates = eligible.groupby('Shipping Mode')['Late_delivery_risk'].agg(eligible_orders='size', late_orders='sum')
rates['late_percent'] = (100 * rates.late_orders / rates.eligible_orders).round(2)
print(rates.sort_values('late_percent', ascending=False).to_string())
print(f'Eligible: {len(eligible):,} | Late: {eligible["Late_delivery_risk"].sum():,} | Shipping canceled: {len(orders)-len(eligible):,}')
first_class = eligible[eligible['Shipping Mode'].eq('First Class')]
print('First Class actual and scheduled days:')
print(first_class[['Days for shipping (real)', 'Days for shipment (scheduled)']].value_counts().to_string())

                eligible_orders  late_orders  late_percent
Shipping Mode                                             
First Class                9602         9602        100.00
Second Class              12256         9803         79.99
Same Day                   3407         1648         48.37
Standard Class            37632        14995         39.85
Eligible: 62,897 | Late: 36,048 | Shipping canceled: 2,855
First Class actual and scheduled days:
Days for shipping (real)  Days for shipment (scheduled)
2                         1                                9602


## 5. Profit by product category
Use the business status COMPLETE for this example. Sum the recorded profit once per item and group by category ID and name. Amounts are shown in the dataset's units. Margin is total profit divided by total discounted sales for the category.

In [6]:
completed = lines[lines['Order Status'].eq('COMPLETE')]
assert completed['Order Item Id'].is_unique
profit = completed.groupby(['Category Id','Category Name']).agg(
    items=('Order Item Id','size'), profit=('Order Profit Per Order','sum'),
    net_sales=('Order Item Total','sum'))
profit['margin_percent'] = (100 * profit.profit / profit.net_sales).round(2)
profit = profit.sort_values('profit', ascending=False)
print(f'COMPLETE: {len(completed):,} items across {completed["Order Id"].nunique():,} orders')
print(profit.head(5).round(2).to_string())

COMPLETE: 59,491 items across 21,716 orders
                              items     profit   net_sales  margin_percent
Category Id Category Name                                                 
45          Fishing            5645  248022.93  2027782.20           12.23
17          Cleats             8085  168760.38  1317653.89           12.81
43          Camping & Hiking   4517  135854.84  1215539.24           11.18
9           Cardio Equipment   4117  128035.09  1089536.86           11.75
24          Women's Apparel    7036  116864.24   939951.75           12.43


## 6. Clickstream relevance
The website log records visits. Inspect its schema, date range and repeated rows before deciding whether to include it.

In [7]:
visited = pd.to_datetime(logs['Date'], format=fmt, errors='coerce')
print(f'Web records: {len(logs):,} | Columns: {logs.shape[1]}')
print(', '.join(logs.columns))
print(f'Visit coverage: {visited.min()} through {visited.max()} | Unparsed: {visited.isna().sum()}')
print(f'Empty entries: {logs.isna().sum().sum():,} | Repeated full rows after first: {logs.duplicated().sum():,}')
print('Order ID or item ID in log: ' + str(bool({'Order Id','Order Item Id'} & set(logs.columns))))

Web records: 469,977 | Columns: 8
Product, Category, Date, Month, Hour, Department, ip, url
Visit coverage: 2017-09-01 06:00:00 through 2018-01-31 23:58:00 | Unparsed: 0
Empty entries: 0 | Repeated full rows after first: 3,249
Order ID or item ID in log: False


The log has no order-level key. Joining on product name would give one order item many matching visits. Keep web traffic as an optional, separate input for questions about browsing. The main CSV provides both delivery and profit measures.

## 7. Data dictionary
The CSV export lists every original header, a plain-English meaning, logical type, project use, missing counts and number of distinct non-missing values. Match the publisher's guide after trimming spaces and ignoring case.

In [8]:
field_notes = {
    'Type': ('Category', 'Payment method recorded for the transaction.', 'Optional payment analysis'),
    'Days for shipping (real)': ('Whole days', 'Recorded actual shipping duration in days.', 'Validate late deliveries'),
    'Days for shipment (scheduled)': ('Whole days', 'Scheduled shipping duration in days.', 'Validate late deliveries'),
    'Benefit per order': ('Decimal amount', 'Recorded profit on this item row. Matches Order Profit Per Order.', 'Alias; use one profit field'),
    'Sales per customer': ('Decimal amount', 'Discounted sales recorded on this row. Matches Order Item Total.', 'Alias; use one net sales field'),
    'Delivery Status': ('Category', 'Recorded shipment outcome, including cancellation.', 'Define eligible and late orders'),
    'Late_delivery_risk': ('Binary flag', '1 marks a late delivery in this dataset, 0 otherwise. Cancellations also carry 0.', 'Late numerator after status filter'),
    'Category Id': ('Identifier', 'Numeric key for the product category.', 'Group category results'),
    'Category Name': ('Text', 'Category display name. Electronics is shared by IDs 13 and 37.', 'Label category results'),
    'Customer City': ('Text', 'City recorded on the customer side of the transaction.', 'Optional customer geography'),
    'Customer Country': ('Text', 'Country recorded for the customer.', 'Optional customer geography'),
    'Customer Email': ('Text', 'Customer email field with one masked value across the file.', 'Omit from analytical view'),
    'Customer Fname': ('Text', 'Customer given name.', 'Omit from analytical view'),
    'Customer Id': ('Identifier', 'Customer identifier repeated across that customer’s transactions.', 'Optional customer counts'),
    'Customer Lname': ('Text', 'Customer family name.', 'Omit from analytical view'),
    'Customer Password': ('Text', 'Masked customer password field with one stored value.', 'Omit from analytical view'),
    'Customer Segment': ('Category', 'Customer group, such as Consumer or Corporate.', 'Optional segmentation'),
    'Customer State': ('Text', 'State recorded on the customer or store side of the transaction.', 'Optional geography'),
    'Customer Street': ('Text', 'Street address associated with the customer.', 'Omit from analytical view'),
    'Customer Zipcode': ('Postal code (text)', 'Postal code associated with the customer.', 'Optional geography'),
    'Department Id': ('Identifier', 'Store department identifier.', 'Optional department grouping'),
    'Department Name': ('Text', 'Store department label.', 'Optional department grouping'),
    'Latitude': ('Decimal coordinate', 'Store latitude, in degrees, according to the publisher.', 'Optional store geography'),
    'Longitude': ('Decimal coordinate', 'Store longitude, in degrees, according to the publisher.', 'Optional store geography'),
    'Market': ('Category', 'Market to which the order is delivered.', 'Optional market comparison'),
    'Order City': ('Text', 'Order destination city.', 'Optional delivery geography'),
    'Order Country': ('Text', 'Order destination country.', 'Optional delivery geography'),
    'Order Customer Id': ('Identifier', 'Customer identifier on the order record. Matches Customer Id.', 'Optional customer linkage'),
    'order date (DateOrders)': ('Timestamp', 'Date and time when the order was placed.', 'Historical order trends'),
    'Order Id': ('Identifier', 'Order key shared by all items in the same order.', 'Count orders once'),
    'Order Item Cardprod Id': ('Identifier', 'Product identifier on the order item. Matches Product Card Id.', 'Optional product linkage'),
    'Order Item Discount': ('Decimal amount', 'Discount amount for the order item.', 'Optional sales reconciliation'),
    'Order Item Discount Rate': ('Decimal fraction', 'Discount rate applied to the item, stored as a fraction.', 'Optional discount analysis'),
    'Order Item Id': ('Identifier', 'Unique identifier for a single order item.', 'Item key and duplicate check'),
    'Order Item Product Price': ('Decimal amount', 'Unit price on the order item before discount.', 'Optional sales reconciliation'),
    'Order Item Profit Ratio': ('Decimal fraction', 'Profit ratio recorded on the order item.', 'Optional item margin comparison'),
    'Order Item Quantity': ('Whole count', 'Units purchased on this order item.', 'Optional sales volume analysis'),
    'Sales': ('Decimal amount', 'Item sales amount before discount.', 'Gross sales analysis'),
    'Order Item Total': ('Decimal amount', 'Item sales value after discount.', 'Net sales and margin denominator'),
    'Order Profit Per Order': ('Decimal amount', 'Profit observed at item level, including negative values.', 'Sum once per item for category profit'),
    'Order Region': ('Category', 'Destination region for the order.', 'Optional delivery breakdown'),
    'Order State': ('Text', 'Destination state or administrative area.', 'Optional delivery geography'),
    'Order Status': ('Category', 'Business status of the order, such as COMPLETE or PENDING.', 'Filter completed-order profit'),
    'Order Zipcode': ('Postal code (text)', 'Postal code for the order destination.', 'Sparse optional geography'),
    'Product Card Id': ('Identifier', 'Product catalogue identifier.', 'Optional product grouping'),
    'Product Category Id': ('Identifier', 'Category key on the product. Matches Category Id.', 'Alias for category grouping'),
    'Product Description': ('Text (empty)', 'Publisher-provided product description field. All values are missing.', 'Omit from analytical view'),
    'Product Image': ('URL', 'Product image link.', 'Omit from analytical view'),
    'Product Name': ('Text', 'Product display name in the catalogue.', 'Optional product labels'),
    'Product Price': ('Decimal amount', 'Listed product price.', 'Optional price comparison'),
    'Product Status': ('Binary flag', 'Publisher stock flag: 0 available, 1 unavailable. Only one value occurs here.', 'No variation for stock analysis'),
    'shipping date (DateOrders)': ('Timestamp', 'Date and time recorded for shipment.', 'Historical shipping trends'),
    'Shipping Mode': ('Category', 'Shipping service level selected for the order.', 'Group delivery results'),
}

In [9]:
guide_names = guide['FIELDS'].str.strip().str.casefold()
undocumented = [c for c in lines if c.strip().casefold() not in set(guide_names)]
print(f'Publisher guide: {len(guide)} entries. Extra main-file column: {undocumented}')
assert undocumented == ['Order Zipcode']
assert len(field_notes) == len(lines.columns) and set(lines.columns) == set(field_notes)
records = []
for col in lines.columns:
    logical_type, meaning, use = field_notes[col]
    records.append({'Column': col, 'Logical type': logical_type, 'Meaning': meaning,
                    'Project use': use, 'Definition basis': ('Inferred from header and values' if col in undocumented else 'Publisher guide and observed data'),
                    'Missing rows': int(missing[col]),
                    'Missing (%)': round(100 * missing[col] / len(lines), 4),
                    'Distinct populated values': int(lines[col].nunique())})
dictionary = pd.DataFrame(records)
dictionary.to_csv('DataCo_Data_Dictionary.csv', index=False, encoding='utf-8-sig')
print(f'Exported all {len(dictionary)} columns to DataCo_Data_Dictionary.csv.')

Publisher guide: 52 entries. Extra main-file column: ['Order Zipcode']
Exported all 53 columns to DataCo_Data_Dictionary.csv.
